# 🚀 Huấn Luyện PhoBERT-DualHead-BiLSTM-CRF Trên Google Colab (GPU Tesla T4)
**Đề tài:** Nhận diện chuỗi ngôn ngữ xúc phạm tiếng Việt (ViHOS)  
**Kiến trúc Đề xuất Mới:** Dual-Head Multi-Task Learning (Token-level Span Head + Clause/Sentence Intent Head)  
**Mục tiêu cốt lõi:** Triệt tiêu hoàn toàn lỗi Báo động giả (False Positive) ở các câu ngữ cảnh trung tính/khen ngợi (Ví dụ: *"Con chó này đẹp, Mày đúng là con chó"*).  
**Môi trường:** Google Colab GPU Tesla T4 (16GB VRAM), PyTorch, HuggingFace Transformers, Seqeval  


## Bước 1: Kiểm tra cấu hình GPU Tesla T4
Vào **Runtime -> Change runtime type -> Chọn T4 GPU** trước khi chạy!

In [ ]:
!nvidia-smi

## Bước 2: Tải Mã Nguồn & Cài đặt thư viện
Tải mã nguồn dự án chứa đầy đủ kiến trúc Dual-Head và bộ dữ liệu ViHOS.

In [ ]:
import os
%cd /content
if os.path.exists('/content/NLP'):
    !rm -rf /content/NLP

# Clone mã nguồn dự án
!git clone https://github.com/barackvn/NLP.git
%cd /content/NLP

# Cài đặt các thư viện cần thiết
!pip install -q transformers pyvi seqeval accelerate

## Bước 3: Kết nối Google Drive để lưu Checkpoint vĩnh viễn

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/ViHOS_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Thư mục lưu checkpoint Drive: {CHECKPOINT_DIR}")

## Bước 4: Huấn Luyện Mô Hình PhoBERT-DualHead-BiLSTM-CRF Multi-Task
- **Hàm Loss:** $\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{CRF}} + 0.5 \times \mathcal{L}_{\text{BCE}}$ (Tối ưu đồng thời Span Tagging và Sentence Intent)
- **Epochs:** 5, **Batch Size:** 16, **PhoBERT LR:** `2e-5`, **Head LR:** `1e-3`
- **Thời gian chạy:** ~18 phút trên GPU Tesla T4.

In [ ]:
!python -m src.train \
    --model_type phobert_dualhead_bilstm_crf \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path {CHECKPOINT_DIR}/best_phobert_dualhead_bilstm_crf.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3 \
    --max_length 128 \
    --patience 3

## Bước 5: Đánh giá mô hình trên tập Test ViHOS (1.106 câu)
Đo lường Span-Precision, Span-Recall, Span-F1 chuẩn IOB2.

In [ ]:
!python -m src.evaluate \
    --model_type phobert_dualhead_bilstm_crf \
    --checkpoint {CHECKPOINT_DIR}/best_phobert_dualhead_bilstm_crf.pt \
    --test_path data/processed/test.json

## Bước 6: Kiểm thử thực tế các câu bẫy ngữ cảnh (Live Inference Verification)

In [ ]:
import sys
sys.path.insert(0, '/content/NLP')
from backend.inference import ViHOSInferenceEngine

engine = ViHOSInferenceEngine(f"{CHECKPOINT_DIR}/best_phobert_dualhead_bilstm_crf.pt")

test_cases = [
    "Con chó này đẹp, Mày đúng là con chó",
    "Hôm nay trời nóng như chó điên",
    "Thằng clm này xàm ngôn vcl",
    "Mày đúng là thứ ngu dốt vô tích sự",
    "Con mèo nhà mình rất khôn và đáng yêu"
]

print('='*70)
for text in test_cases:
    res = engine.predict(text, enable_gated_intent=True)
    print(f"CÂU GỐC: {text}")
    print(f"  -> Spans bắt được   : {res['spans']}")
    print(f"  -> Auto-Masking (***): {engine.auto_mask(res['words'], res['tags'])}")
    print(f"  -> Gated Filter     : {res['gated_filter_applied']} ({res['gated_note']})")
    print('-'*70)

## Bước 7: Tải Checkpoint về máy tính cá nhân để nạp vào Web App

In [ ]:
from google.colab import files

# Tải file checkpoint về thư mục Doan/checkpoints/ trên máy tính
files.download(f"{CHECKPOINT_DIR}/best_phobert_dualhead_bilstm_crf.pt")